# LeetCode 1103 — Distribute Candies to People bằng Python

**Link đề:** https://leetcode.com/problems/distribute-candies-to-people/
**Độ khó:** Easy · **Chủ đề:** Math, Simulation

```python
class Solution:
    def distributeCandies(self, candies: int, num_people: int) -> List[int]:
        ...
```

## Mục tiêu học
1. Đọc kỹ một đề **mô phỏng** (simulation): ai nhận, nhận bao nhiêu, khi nào dừng.
2. Dùng `list` khởi tạo `[0] * n`, chỉ số vòng tròn `i % n` và `min()` để viết mô phỏng gọn, đúng.
3. Hiểu vì sao mô phỏng chỉ tốn khoảng `√(2·candies)` bước, không phải `candies` bước.
4. Nâng cấp bằng **toán học**: tổng cấp số cộng `1 + 2 + ... + p = p(p+1)/2`, `math.isqrt`, chia vòng đầy đủ và phần dư.
5. Trace từng bước, chứng minh tính đúng, phân tích Time/Space, kiểm thử đối chiếu, tự giải biến thể.

**Cách học:** đọc markdown → đoán output → chạy cell → so sánh. Cell TODO chỉ để luyện, không tự chạy.

## 1. Đề bài — chia kẹo theo hàng, mỗi lượt tăng 1 viên

Có `candies` viên kẹo và `n = num_people` người xếp thành hàng. Cách chia:

- Lượt đầu: người 1 nhận `1`, người 2 nhận `2`, …, người `n` nhận `n`.
- Quay lại đầu hàng: người 1 nhận `n+1`, người 2 nhận `n+2`, …, người `n` nhận `2n`.
- Cứ thế, **mỗi lần trao tăng thêm 1 viên** và quay vòng, cho tới khi hết kẹo. **Người cuối cùng nhận toàn bộ số kẹo còn lại** (có thể ít hơn số đáng lẽ được nhận).

Trả về mảng độ dài `num_people`, tổng bằng `candies`, cho biết mỗi người nhận bao nhiêu.

| candies | num_people | Output | Diễn giải |
|---:|---:|---|---|
| 7 | 4 | `[1,2,3,1]` | trao 1,2,3 → còn 1 → người 4 nhận 1 |
| 10 | 3 | `[5,2,3]` | trao 1,2,3 → còn 4 → người 1 nhận 4 (đủ) → tổng người 1 là 5 |

**Constraints:** `1 <= candies <= 10^9`, `1 <= num_people <= 1000`.

**Gợi ý của đề:** "Chia cho mọi người theo từng vòng trước (khi còn đủ), rồi chia từng người một."

In [1]:
# Kiểm tra bằng tay ví dụ 2: candies = 10, n = 3
ans = [0, 0, 0]
ans[0] += 1; ans[1] += 2; ans[2] += 3      # vòng 1: dùng 6 viên, còn 4
print("Sau vòng 1:", ans, "| còn", 10 - 6)
ans[0] += 4                                # lượt kế: người 1 cần 4, còn đúng 4
print("Sau lượt 4:", ans, "| tổng =", sum(ans))

Sau vòng 1: [1, 2, 3] | còn 4
Sau lượt 4: [5, 2, 3] | tổng = 10


## 2. Nền tảng Python cần dùng

### 2.1. Tạo list toàn số 0: `[0] * n`

`[0] * 4` tạo list `[0, 0, 0, 0]`. Mỗi phần tử là số kẹo một người đang có. Ta cộng dồn bằng `ans[i] += x` (viết tắt của `ans[i] = ans[i] + x`).

In [2]:
ans = [0] * 4
print(ans)
ans[2] += 5
ans[2] += 1
print(ans, "| len =", len(ans))

[0, 0, 0, 0]
[0, 0, 6, 0] | len = 4


### 2.2. Chỉ số vòng tròn: `i % n`

Khi đi hết hàng phải quay về đầu. Nếu đếm số lần trao là `t = 0, 1, 2, ...` thì người nhận là `t % n` (phần dư khi chia cho `n`):

- `n = 3`: `t = 0,1,2,3,4,5,6` → người `0,1,2,0,1,2,0`.

`%` là phép **chia lấy dư** (modulo). `//` là **chia lấy phần nguyên**. Với `t = 7, n = 3`: `7 // 3 = 2` (đã đi xong 2 vòng), `7 % 3 = 1` (đang ở người thứ 1 của vòng 3).

In [3]:
n = 3
for t in range(8):
    print(f"lần trao t={t}: người {t % n}, thuộc vòng {t // n}")

lần trao t=0: người 0, thuộc vòng 0
lần trao t=1: người 1, thuộc vòng 0
lần trao t=2: người 2, thuộc vòng 0
lần trao t=3: người 0, thuộc vòng 1
lần trao t=4: người 1, thuộc vòng 1
lần trao t=5: người 2, thuộc vòng 1
lần trao t=6: người 0, thuộc vòng 2
lần trao t=7: người 1, thuộc vòng 2


### 2.3. `min(a, b)` — trao phần còn lại

Lần trao thứ `t` (đếm từ 0) đáng lẽ trao `t + 1` viên. Nhưng nếu kẹo còn ít hơn thì chỉ trao được phần còn lại. Gộp hai trường hợp: `give = min(t + 1, candies)`.

In [4]:
for need, left in [(4, 10), (4, 4), (4, 1)]:
    print(f"cần trao {need}, còn {left} -> trao {min(need, left)}")

cần trao 4, còn 10 -> trao 4
cần trao 4, còn 4 -> trao 4
cần trao 4, còn 1 -> trao 1


## 3. Cách 1 — Mô phỏng từng lần trao (trực giác)

Làm đúng như đề mô tả:

```text
ans = [0] * n
t = 0                       # lần trao thứ t, trao t+1 viên
while candies > 0:
    give = min(t + 1, candies)
    ans[t % n] += give
    candies -= give
    t += 1
```

In [5]:
from typing import List

def distribute_simulation(candies: int, num_people: int) -> List[int]:
    ans = [0] * num_people
    t = 0
    while candies > 0:
        give = min(t + 1, candies)
        ans[t % num_people] += give
        candies -= give
        t += 1
    return ans

print(distribute_simulation(7, 4))    # [1, 2, 3, 1]
print(distribute_simulation(10, 3))   # [5, 2, 3]
print(distribute_simulation(1, 5))    # [1, 0, 0, 0, 0]

[1, 2, 3, 1]
[5, 2, 3]
[1, 0, 0, 0, 0]


### 3.1. Mô phỏng chạy bao nhiêu bước?

Sau `p` lần trao đầy đủ, đã dùng `1 + 2 + ... + p = p(p+1)/2` viên. Vòng lặp dừng khi tổng này vượt `candies`, tức `p ≈ √(2·candies)`.

- `candies = 10^9` → khoảng `44 721` bước. Rất nhanh!
- Nhưng nếu ai đó viết mô phỏng **từng viên kẹo** (mỗi bước trao 1 viên) thì cần `10^9` bước → quá chậm.

Đây là điểm thú vị: mô phỏng "đúng đơn vị" (mỗi lần trao) đã đủ nhanh.

In [6]:
def count_steps(candies: int, num_people: int) -> int:
    t = 0
    while candies > 0:
        candies -= min(t + 1, candies)
        t += 1
    return t

import math
for c in [10, 1000, 10**6, 10**9]:
    print(f"candies={c:>10}: số lần trao = {count_steps(c, 7):>6}, √(2c) ≈ {math.sqrt(2 * c):,.0f}")

candies=        10: số lần trao =      4, √(2c) ≈ 4
candies=      1000: số lần trao =     45, √(2c) ≈ 45
candies=   1000000: số lần trao =   1414, √(2c) ≈ 1,414
candies=1000000000: số lần trao =  44721, √(2c) ≈ 44,721


**Độ phức tạp cách 1:** Time `O(√candies + n)` (cộng `n` để tạo mảng), Space `O(n)` cho mảng kết quả (bắt buộc phải trả về) — bộ nhớ phụ `O(1)`.

**Nút thắt?** Với constraints này thì không có nút thắt thực sự. Nhưng bài cho tag **Math**: có thể tính trực tiếp mà không cần lặp tới `√candies` bước. Học cách toán học giúp ta xử lý được phiên bản lớn hơn (ví dụ `candies = 10^18`).

## 4. Nền tảng toán: tổng cấp số cộng

**Cấp số cộng** (arithmetic progression): dãy số cách đều, ví dụ `1, 2, 3, ..., p`.

Công thức: `1 + 2 + ... + p = p(p+1)/2`. Cách nhớ (Gauss): ghép số đầu với số cuối `1 + p`, `2 + (p-1)`, ... mỗi cặp bằng `p+1`, có `p/2` cặp.

Tổng quát: dãy cách đều có `m` số, số đầu `a`, số cuối `z` → tổng `= m · (a + z) / 2`.

In [7]:
p = 10
print("Cộng trực tiếp:", sum(range(1, p + 1)))
print("Công thức     :", p * (p + 1) // 2)

# Dãy cách đều: người 1 với n = 3 nhận 1, 4, 7, 10 (mỗi vòng tăng n = 3)
seq = [1, 4, 7, 10]
m, a, z = len(seq), seq[0], seq[-1]
print("Tổng dãy", seq, "=", sum(seq), "| công thức m(a+z)/2 =", m * (a + z) // 2)

Cộng trực tiếp: 55
Công thức     : 55
Tổng dãy [1, 4, 7, 10] = 22 | công thức m(a+z)/2 = 22


### 4.1. `math.isqrt` — căn bậc hai nguyên chính xác

Ta cần `p` lớn nhất sao cho `p(p+1)/2 <= candies`. Giải bất phương trình bậc hai: `p <= (√(8·candies + 1) - 1) / 2`.

`math.sqrt` trả số thực, có thể sai lệch nhỏ với số lớn. `math.isqrt(x)` trả **phần nguyên của căn** chính xác tuyệt đối với số nguyên. Sau khi tính, ta vẫn "chỉnh" thêm bằng vòng `while` nhỏ cho chắc chắn.

In [8]:
import math
print(math.isqrt(15), math.isqrt(16), math.isqrt(17))   # 3 4 4
c = 7
p = (math.isqrt(8 * c + 1) - 1) // 2
print("candies =", c, "-> số lần trao đầy đủ p =", p, "| dùng", p * (p + 1) // 2, "viên")
c = 10**9
p = (math.isqrt(8 * c + 1) - 1) // 2
print("candies = 10^9 -> p =", p, "| p(p+1)/2 =", p * (p + 1) // 2, "| (p+1)(p+2)/2 =", (p + 1) * (p + 2) // 2)

3 4 4
candies = 7 -> số lần trao đầy đủ p = 3 | dùng 6 viên
candies = 10^9 -> p = 44720 | p(p+1)/2 = 999961560 | (p+1)(p+2)/2 = 1000006281


## 5. Insight — tách thành "trao đầy đủ" + "phần dư"

1. Tính `p` = số lần trao **đầy đủ** (trao đúng `t+1` viên). Kẹo đã dùng `p(p+1)/2`, phần dư `rem = candies - p(p+1)/2` (có thể bằng 0) được trao ở lần thứ `p` (đếm từ 0) cho người `p % n`.
2. `p` lần trao đầy đủ = `rows = p // n` **vòng trọn vẹn** + `cols = p % n` lần trao lẻ ở vòng cuối (cho người `0..cols-1`).
3. Người `i` (0-indexed) ở vòng `r` (0-indexed) nhận `r·n + (i + 1)`.
   - Qua `rows` vòng trọn: nhận `Σ_{r=0}^{rows-1} (r·n + i + 1) = rows·(i+1) + n·rows·(rows-1)/2`.
   - Nếu `i < cols`: nhận thêm `rows·n + (i + 1)` ở vòng cuối dở dang.
4. Cộng `rem` cho người `p % n`.

In [9]:
# Áp dụng tay cho candies = 10, n = 3
candies, n = 10, 3
p = (math.isqrt(8 * candies + 1) - 1) // 2
rows, cols = divmod(p, n)          # divmod trả (p // n, p % n)
rem = candies - p * (p + 1) // 2
print("p =", p, "| rows =", rows, "| cols =", cols, "| rem =", rem, "-> người", p % n)

p = 4 | rows = 1 | cols = 1 | rem = 0 -> người 1


`divmod(a, b)` trả về cặp `(a // b, a % b)` trong một lần gọi — tiện khi cần cả thương và dư.

## 6. Cách 2 — Mô phỏng theo từng **vòng** (theo gợi ý của đề)

Trước khi tới công thức đóng hoàn toàn, có một bước trung gian: trao cả vòng một lần khi còn đủ kẹo.

Vòng `r` (0-indexed) tốn `Σ_{i=1}^{n} (r·n + i) = r·n² + n(n+1)/2` viên. Khi không còn đủ cho cả vòng, chuyển sang trao từng người.

Số vòng trọn vẹn khoảng `√(2·candies)/n`, nên cách này nhanh hơn cách 1 khi `n` lớn.

In [10]:
def distribute_by_rounds(candies: int, num_people: int) -> List[int]:
    n = num_people
    ans = [0] * n
    r = 0
    # 1) trao trọn vẹn từng vòng khi còn đủ
    while candies >= r * n * n + n * (n + 1) // 2:
        candies -= r * n * n + n * (n + 1) // 2
        r += 1
    # mỗi người i đã nhận qua r vòng: r*(i+1) + n*r*(r-1)/2
    for i in range(n):
        ans[i] = r * (i + 1) + n * r * (r - 1) // 2
    # 2) vòng dở dang: trao từng người
    i = 0
    while candies > 0:
        give = min(r * n + i + 1, candies)
        ans[i] += give
        candies -= give
        i += 1
    return ans

print(distribute_by_rounds(7, 4))    # [1, 2, 3, 1]
print(distribute_by_rounds(10, 3))   # [5, 2, 3]
print(distribute_by_rounds(60, 3))
print(distribute_simulation(60, 3))

[1, 2, 3, 1]
[5, 2, 3]
[22, 20, 18]
[22, 20, 18]


**Độ phức tạp cách 2:** số vòng trọn `R ≈ √(2·candies)/n`, cộng `O(n)` cho vòng dở dang → Time `O(R + n)`; Space phụ `O(1)`.

## 7. Cách 3 — Công thức đóng (lời giải chính, Math)

Ghép các mảnh ở mục 5:

In [11]:
def distribute_math(candies: int, num_people: int) -> List[int]:
    n = num_people
    p = (math.isqrt(8 * candies + 1) - 1) // 2      # số lần trao đầy đủ
    while (p + 1) * (p + 2) // 2 <= candies:        # chỉnh phòng sai số (thường không chạy)
        p += 1
    while p * (p + 1) // 2 > candies:
        p -= 1
    rows, cols = divmod(p, n)
    ans = []
    for i in range(n):
        got = rows * (i + 1) + n * rows * (rows - 1) // 2   # qua các vòng trọn vẹn
        if i < cols:
            got += rows * n + (i + 1)                       # lần trao ở vòng dở dang
        ans.append(got)
    ans[p % n] += candies - p * (p + 1) // 2                # phần dư cho người kế tiếp
    return ans

print(distribute_math(7, 4))     # [1, 2, 3, 1]
print(distribute_math(10, 3))    # [5, 2, 3]
print(distribute_math(10**9, 1000)[:5], "...")

[1, 2, 3, 1]
[5, 2, 3]
[990045, 990090, 990135, 990180, 990225] ...


### 7.1. Vì sao phần dư luôn trao cho người `p % n`?

Các lần trao được đánh số `0, 1, ..., p-1` là đầy đủ. Lần trao tiếp theo có số thứ tự `p` → người nhận `p % n`. Lần đó cần `p + 1` viên nhưng chỉ còn `rem < p + 1` (theo định nghĩa `p` lớn nhất). Nếu `rem = 0` thì cộng 0, không ảnh hưởng.

## 8. TRACE từng bước

### 8.1. Trace cách mô phỏng — `candies = 10, n = 3`

| t | người `t % n` | cần `t+1` | còn trước | trao | còn sau | ans |
|---:|---:|---:|---:|---:|---:|---|
| 0 | 0 | 1 | 10 | 1 | 9 | `[1,0,0]` |
| 1 | 1 | 2 | 9 | 2 | 7 | `[1,2,0]` |
| 2 | 2 | 3 | 7 | 3 | 4 | `[1,2,3]` |
| 3 | 0 | 4 | 4 | 4 | 0 | `[5,2,3]` |

Dừng vì `còn = 0`.

In [12]:
def trace_simulation(candies: int, num_people: int) -> List[int]:
    ans = [0] * num_people
    t = 0
    print(f"candies={candies}, n={num_people}")
    while candies > 0:
        need = t + 1
        give = min(need, candies)
        who = t % num_people
        before = candies
        ans[who] += give
        candies -= give
        print(f"t={t:>2} | người {who} | cần {need:>2} | còn trước {before:>3} | trao {give:>2} | còn sau {candies:>3} | ans={ans}")
        t += 1
    return ans

_ = trace_simulation(10, 3)
print()
_ = trace_simulation(20, 4)

candies=10, n=3
t= 0 | người 0 | cần  1 | còn trước  10 | trao  1 | còn sau   9 | ans=[1, 0, 0]
t= 1 | người 1 | cần  2 | còn trước   9 | trao  2 | còn sau   7 | ans=[1, 2, 0]
t= 2 | người 2 | cần  3 | còn trước   7 | trao  3 | còn sau   4 | ans=[1, 2, 3]
t= 3 | người 0 | cần  4 | còn trước   4 | trao  4 | còn sau   0 | ans=[5, 2, 3]

candies=20, n=4
t= 0 | người 0 | cần  1 | còn trước  20 | trao  1 | còn sau  19 | ans=[1, 0, 0, 0]
t= 1 | người 1 | cần  2 | còn trước  19 | trao  2 | còn sau  17 | ans=[1, 2, 0, 0]
t= 2 | người 2 | cần  3 | còn trước  17 | trao  3 | còn sau  14 | ans=[1, 2, 3, 0]
t= 3 | người 3 | cần  4 | còn trước  14 | trao  4 | còn sau  10 | ans=[1, 2, 3, 4]
t= 4 | người 0 | cần  5 | còn trước  10 | trao  5 | còn sau   5 | ans=[6, 2, 3, 4]
t= 5 | người 1 | cần  6 | còn trước   5 | trao  5 | còn sau   0 | ans=[6, 7, 3, 4]


### 8.2. Trace cách công thức — `candies = 20, n = 4`

- `p`: `5·6/2 = 15 <= 20`, `6·7/2 = 21 > 20` → `p = 5`, `rem = 5`.
- `rows, cols = divmod(5, 4) = (1, 1)`.
- Người `i`: `rows·(i+1) + n·rows·(rows-1)/2 = (i+1) + 0` → `[1,2,3,4]`.
- `i < cols = 1` → người 0 thêm `rows·n + 1 = 5` → `[6,2,3,4]`.
- Phần dư `5` cho người `p % n = 1` → `[6,7,3,4]`. Khớp với trace mô phỏng ở trên.

In [13]:
def trace_math(candies: int, num_people: int) -> List[int]:
    n = num_people
    p = (math.isqrt(8 * candies + 1) - 1) // 2
    rows, cols = divmod(p, n)
    rem = candies - p * (p + 1) // 2
    print(f"p={p}, rows={rows}, cols={cols}, rem={rem} -> người nhận dư: {p % n}")
    ans = []
    for i in range(n):
        full = rows * (i + 1) + n * rows * (rows - 1) // 2
        extra = rows * n + (i + 1) if i < cols else 0
        print(f"  người {i}: qua {rows} vòng trọn = {full}, vòng dở dang = {extra}")
        ans.append(full + extra)
    ans[p % n] += rem
    print("Kết quả:", ans, "| tổng =", sum(ans))
    return ans

_ = trace_math(20, 4)

p=5, rows=1, cols=1, rem=5 -> người nhận dư: 1
  người 0: qua 1 vòng trọn = 1, vòng dở dang = 5
  người 1: qua 1 vòng trọn = 2, vòng dở dang = 0
  người 2: qua 1 vòng trọn = 3, vòng dở dang = 0
  người 3: qua 1 vòng trọn = 4, vòng dở dang = 0
Kết quả: [6, 7, 3, 4] | tổng = 20


## 9. Code nộp LeetCode

Bản mô phỏng (ngắn, dễ nhớ, đủ nhanh):

In [14]:
class SolutionSimulation:
    def distributeCandies(self, candies: int, num_people: int) -> List[int]:
        ans = [0] * num_people
        t = 0
        while candies > 0:
            give = min(t + 1, candies)
            ans[t % num_people] += give
            candies -= give
            t += 1
        return ans

print(SolutionSimulation().distributeCandies(7, 4))

[1, 2, 3, 1]


Bản công thức (Math):

In [15]:
import math
from typing import List

class Solution:
    def distributeCandies(self, candies: int, num_people: int) -> List[int]:
        n = num_people
        p = (math.isqrt(8 * candies + 1) - 1) // 2
        rows, cols = divmod(p, n)
        ans = [0] * n
        for i in range(n):
            ans[i] = rows * (i + 1) + n * rows * (rows - 1) // 2
            if i < cols:
                ans[i] += rows * n + i + 1
        ans[p % n] += candies - p * (p + 1) // 2
        return ans

sol = Solution()
print(sol.distributeCandies(7, 4))    # [1, 2, 3, 1]
print(sol.distributeCandies(10, 3))   # [5, 2, 3]

[1, 2, 3, 1]
[5, 2, 3]


## 10. Vì sao đúng?

**Mô phỏng:** invariant — trước mỗi vòng lặp, `ans` đúng bằng số kẹo mỗi người nhận được sau `t` lần trao, và `candies` là số còn lại. Mỗi vòng thực hiện đúng lần trao thứ `t` của đề (người `t % n`, `min(t+1, còn lại)` viên). Vòng lặp dừng đúng khi hết kẹo; vì mỗi lần trao ít nhất 1 viên nên chắc chắn dừng. Tổng `ans` luôn bằng số kẹo đã trao → cuối cùng bằng `candies` ban đầu.

**Công thức:** `p` là số lần trao đầy đủ duy nhất thỏa `p(p+1)/2 <= candies < (p+1)(p+2)/2`. Lần trao thứ `t` thuộc vòng `t // n`, người `t % n`, lượng `t + 1 = (t // n)·n + (t % n) + 1` — đúng với biểu thức `r·n + i + 1`. Gom theo người: người `i` nhận ở các vòng `r = 0..rows-1` và thêm vòng `rows` nếu `i < cols`. Tổng hợp bằng công thức cấp số cộng. Cuối cùng phần dư `rem` thuộc lần trao thứ `p`.

**Kiểm tra `math.isqrt`:** `p = ⌊(√(8c+1) - 1)/2⌋` là nghiệm nguyên lớn nhất của `p² + p - 2c <= 0`; `isqrt` cho căn nguyên chính xác nên công thức đúng với số nguyên (bản `Solution` không cần vòng chỉnh). Ở mục test ta kiểm chứng điều này trên nhiều `c`.

## 11. Độ phức tạp

Gọi `c = candies`, `n = num_people`.

| Cách | Time | Space phụ (không tính mảng kết quả) |
|---|---|---|
| 1. Mô phỏng từng lần trao | `O(√c + n)` | `O(1)` |
| 2. Mô phỏng theo vòng | `O(√c / n + n)` | `O(1)` |
| 3. Công thức đóng | `O(n)` (+ `isqrt` trên số `c`, coi như `O(1)` với số máy) | `O(1)` |

Với `c = 10^9`: cách 1 khoảng 44 721 bước, cách 3 chỉ `n` bước.

## 12. Bộ test — mẫu đề, biên, phản ví dụ, đối chiếu

In [16]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<28} PASS {actual}")

cases = [
    ("mẫu 1", 7, 4, [1, 2, 3, 1]),
    ("mẫu 2", 10, 3, [5, 2, 3]),
    ("1 viên, 1 người", 1, 1, [1]),
    ("1 người nhận hết", 100, 1, [100]),
    ("kẹo ít hơn số người", 3, 5, [1, 2, 0, 0, 0]),
    ("vừa hết đúng 1 vòng", 6, 3, [1, 2, 3]),
    ("dư = 0 nhiều vòng", 21, 3, [5, 7, 9]),
    ("dư quay về người đầu", 9, 3, [4, 2, 3]),
]
for name, c, n, exp in cases:
    check(name, Solution().distributeCandies(c, n), exp)

mẫu 1                        PASS [1, 2, 3, 1]
mẫu 2                        PASS [5, 2, 3]
1 viên, 1 người              PASS [1]
1 người nhận hết             PASS [100]
kẹo ít hơn số người          PASS [1, 2, 0, 0, 0]
vừa hết đúng 1 vòng          PASS [1, 2, 3]
dư = 0 nhiều vòng            PASS [5, 7, 9]
dư quay về người đầu         PASS [4, 2, 3]


Đối chiếu 4 cách (kể cả bản class) trên mọi `candies` từ 1 tới 600 với nhiều `n`, kiểm tra thêm tổng bằng `candies`, và kiểm tra `p` từ `isqrt` khớp định nghĩa với số rất lớn.

In [17]:
for c in range(1, 601):
    for n in [1, 2, 3, 4, 7, 10, 50]:
        a = distribute_simulation(c, n)
        assert distribute_by_rounds(c, n) == a, (c, n)
        assert distribute_math(c, n) == a, (c, n)
        assert Solution().distributeCandies(c, n) == a, (c, n)
        assert sum(a) == c and len(a) == n
print("Đối chiếu c = 1..600 với 7 giá trị n: PASS")

import random
rng = random.Random(1103)
for _ in range(2000):
    c = rng.randint(1, 10**18)
    p = (math.isqrt(8 * c + 1) - 1) // 2
    assert p * (p + 1) // 2 <= c < (p + 1) * (p + 2) // 2
print("isqrt cho p đúng với 2000 số ngẫu nhiên tới 10^18")
assert sum(Solution().distributeCandies(10**9, 1000)) == 10**9
assert Solution().distributeCandies(10**9, 1000) == distribute_simulation(10**9, 1000)
print("candies = 10^9, n = 1000: khớp mô phỏng")

Đối chiếu c = 1..600 với 7 giá trị n: PASS
isqrt cho p đúng với 2000 số ngẫu nhiên tới 10^18
candies = 10^9, n = 1000: khớp mô phỏng


## 13. Lỗi người mới hay gặp

### Lỗi 1: Trao `t+1` viên mà không kiểm tra còn đủ không

In [18]:
def wrong_no_min(candies, n):
    ans = [0] * n; t = 0
    while candies > 0:
        ans[t % n] += t + 1      # SAI: có thể trao nhiều hơn số còn lại
        candies -= t + 1
        t += 1
    return ans

w = wrong_no_min(7, 4)
print("sai:", w, "tổng", sum(w), "| đúng:", distribute_simulation(7, 4))

sai: [1, 2, 3, 4] tổng 10 | đúng: [1, 2, 3, 1]


**Vì sao sai:** người cuối chỉ nhận phần **còn lại**, nên tổng phải bằng `candies`. Ở đây trao 4 viên trong khi chỉ còn 1 → tổng 10 ≠ 7. **Sửa:** `give = min(t + 1, candies)`.

### Lỗi 2: Quên quay vòng — dùng `t` làm chỉ số trực tiếp

In [19]:
def wrong_index(candies, n):
    ans = [0] * n; t = 0
    try:
        while candies > 0:
            give = min(t + 1, candies)
            ans[t] += give          # SAI: t vượt quá n-1
            candies -= give
            t += 1
    except IndexError as e:
        return f"IndexError: {e}"
    return ans

print(wrong_index(10, 3))

IndexError: list index out of range


**Vì sao sai:** sau `n` lần trao, `t = n` vượt chỉ số cuối `n-1`. **Sửa:** `ans[t % n]`.

### Lỗi 3: Mỗi vòng mới lại trao từ 1

Đọc nhầm đề thành "mỗi vòng bắt đầu lại từ 1". Đề nói lượng trao **tiếp tục tăng**: vòng 2 bắt đầu từ `n + 1`.

In [20]:
def wrong_restart(candies, n):
    ans = [0] * n; t = 0
    while candies > 0:
        give = min(t % n + 1, candies)   # SAI: reset mỗi vòng
        ans[t % n] += give
        candies -= give
        t += 1
    return ans

print("sai:", wrong_restart(10, 3), "| đúng:", distribute_simulation(10, 3))

sai: [2, 4, 4] | đúng: [5, 2, 3]


### Lỗi 4: Dùng `/` hoặc `math.sqrt` trong công thức

`p * (p + 1) / 2` cho số thực (`float`), có thể làm `ans` chứa `6.0` thay vì `6`; với số rất lớn còn sai do làm tròn. **Sửa:** dùng `//` và `math.isqrt`.

In [21]:
p = 5
print(p * (p + 1) / 2, type(p * (p + 1) / 2))
print(p * (p + 1) // 2, type(p * (p + 1) // 2))
big = 10**30
print("sqrt float:", int(math.sqrt(big * big + 1)), "| isqrt:", math.isqrt(big * big + 1))

15.0 <class 'float'>
15 <class 'int'>
sqrt float: 1000000000000000019884624838656 | isqrt: 1000000000000000000000000000000


## 14. Mẹo nhận dạng pattern

- Đề mô tả **quy trình lặp đi lặp lại** (trao, xoay vòng, bước nhảy) → bắt đầu bằng **mô phỏng**, nhưng ước lượng số bước trước khi code.
- Lượng thay đổi **tăng đều 1** mỗi bước → tổng là cấp số cộng `p(p+1)/2` → số bước `~√(2c)`.
- "Quay về đầu hàng" → chỉ số `t % n`, số vòng `t // n` (`divmod`).
- Muốn bỏ vòng lặp → tách "phần trọn vẹn" (công thức) + "phần dư" (xử lý riêng).
- Bài cùng họ: #441 Arranging Coins (tìm `p` lớn nhất với `p(p+1)/2 <= n`), #1518 Water Bottles, #1688 Count of Matches, #2079 Watering Plants.

## 15. Biến thể

### 15.1. Arranging Coins (#441): có bao nhiêu hàng trọn vẹn?

Chính là bước tính `p` của ta.

In [22]:
def arrange_coins(c: int) -> int:
    return (math.isqrt(8 * c + 1) - 1) // 2

print([arrange_coins(c) for c in [1, 3, 5, 6, 8, 10**9]])

[1, 2, 2, 3, 3, 44720]


### 15.2. Mỗi lần trao tăng `d` viên (thay vì 1) — mô phỏng tổng quát

Lần trao thứ `t` trao `1 + t·d` viên. Mô phỏng chỉ đổi đúng một biểu thức; với `d = 1` phải ra kết quả gốc.

In [23]:
def distribute_step_d(candies: int, n: int, d: int) -> List[int]:
    ans = [0] * n
    t = 0
    while candies > 0:
        give = min(1 + t * d, candies)
        ans[t % n] += give
        candies -= give
        t += 1
    return ans

print(distribute_step_d(10, 3, 1), "(d=1, giống đề gốc)")
print(distribute_step_d(20, 3, 2), "(d=2: trao 1,3,5,7,...)")

[5, 2, 3] (d=1, giống đề gốc)
[8, 7, 5] (d=2: trao 1,3,5,7,...)


## 16. Bài tập (tăng dần)

**Bài 1 — Tính tay:** `candies = 15`, `num_people = 2`. Lập bảng trace mô phỏng, kết quả là gì?

**Bài 2 — Dự đoán:** với `candies = 28, n = 7`, tính `p`, `rows`, `cols`, `rem`. Kết quả cuối là gì?

**Bài 3 — TODO:** viết mô phỏng.

In [24]:
def exercise_distribute(candies: int, num_people: int) -> List[int]:
    # TODO 1: ans = [0] * num_people, t = 0
    # TODO 2: while candies > 0: give = min(...), cộng vào ans[t % n], trừ candies, t += 1
    pass

# assert exercise_distribute(7, 4) == [1, 2, 3, 1]
# assert exercise_distribute(10, 3) == [5, 2, 3]

**Bài 4 — Người nhận cuối:** trả về chỉ số người nhận **lần trao cuối cùng** (người lấy phần dư hoặc lần trao đầy đủ cuối nếu vừa hết). Làm bằng công thức, không mô phỏng.

In [25]:
def exercise_last_receiver(candies: int, num_people: int) -> int:
    # TODO: dùng p và rem
    pass

# assert exercise_last_receiver(7, 4) == 3
# assert exercise_last_receiver(6, 3) == 2

**Bài 5 — Số kẹo của một người:** chỉ tính `ans[i]` cho **một** người `i` mà không tạo cả mảng (Time `O(1)`).

In [26]:
def exercise_candies_of(candies: int, num_people: int, i: int) -> int:
    # TODO
    pass

# assert exercise_candies_of(10, 3, 0) == 5
# assert exercise_candies_of(20, 4, 1) == 7

**Bài 6 — Mini challenge:** viết bản công thức cho biến thể "mỗi lần trao tăng `d`" (mục 15.2). Gợi ý: tổng `p` lần trao đầy đủ là `p + d·p(p-1)/2`; tìm `p` bằng tìm nhị phân.

In [27]:
def exercise_distribute_step_d_math(candies: int, n: int, d: int) -> List[int]:
    # TODO
    pass

# assert exercise_distribute_step_d_math(20, 3, 2) == distribute_step_d(20, 3, 2)

### 16.1. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | Trao 1,2,3,... luân phiên 2 người | Người 0 nhận lần t chẵn | 1+3+5 và 2+4 rồi dư |
| 2 | `7·8/2 = 28` | `p = 7`, `divmod(7,7)` | `rows=1, cols=0, rem=0` → `[1..7]` |
| 3 | Làm như mục 3 | `min(t + 1, candies)` | `ans[t % num_people] += give` |
| 4 | Lần trao cuối có số thứ tự `p` nếu `rem > 0` | nếu `rem == 0` thì là `p - 1` | `return (p if rem else p - 1) % n` |
| 5 | Dùng công thức trong vòng `for` của cách 3 | chỉ thay `i` cố định | thêm `rem` nếu `i == p % n` |
| 6 | Tổng đầy đủ tăng đơn điệu theo `p` | tìm nhị phân `p` lớn nhất | lần trao `t` → vòng `t//n`, lượng `1 + t·d` |

## 17. Đáp án tham khảo

- **Bài 1:** trao `1,2,3,4,5` (tổng 15) → người 0: `1+3+5 = 9`, người 1: `2+4 = 6` → `[9, 6]`.
- **Bài 2:** `p = 7` (`28 = 7·8/2`), `rows = 1`, `cols = 0`, `rem = 0` → `[1,2,3,4,5,6,7]`.

In [28]:
def answer_distribute(candies, num_people):
    ans = [0] * num_people; t = 0
    while candies > 0:
        give = min(t + 1, candies)
        ans[t % num_people] += give
        candies -= give
        t += 1
    return ans

def answer_last_receiver(candies, num_people):
    p = (math.isqrt(8 * candies + 1) - 1) // 2
    rem = candies - p * (p + 1) // 2
    return (p if rem else p - 1) % num_people

def answer_candies_of(candies, num_people, i):
    n = num_people
    p = (math.isqrt(8 * candies + 1) - 1) // 2
    rows, cols = divmod(p, n)
    got = rows * (i + 1) + n * rows * (rows - 1) // 2
    if i < cols:
        got += rows * n + i + 1
    if i == p % n:
        got += candies - p * (p + 1) // 2
    return got

def answer_distribute_step_d_math(candies, n, d):
    full = lambda p: p + d * p * (p - 1) // 2      # tổng p lần trao đầy đủ
    lo, hi = 0, 1
    while full(hi) <= candies:
        hi *= 2
    while lo < hi:                                 # p lớn nhất với full(p) <= candies
        mid = (lo + hi + 1) // 2
        if full(mid) <= candies:
            lo = mid
        else:
            hi = mid - 1
    p = lo
    rows, cols = divmod(p, n)
    ans = []
    for i in range(n):
        # lần trao t = r*n + i, lượng 1 + t*d, r = 0..rows-1
        got = rows * (1 + i * d) + d * n * rows * (rows - 1) // 2
        if i < cols:
            got += 1 + (rows * n + i) * d
        ans.append(got)
    ans[p % n] += candies - full(p)
    return ans

assert answer_distribute(15, 2) == [9, 6]
assert answer_distribute(28, 7) == [1, 2, 3, 4, 5, 6, 7]
assert answer_distribute(7, 4) == [1, 2, 3, 1]
assert answer_last_receiver(7, 4) == 3
assert answer_last_receiver(6, 3) == 2
assert answer_candies_of(10, 3, 0) == 5
assert answer_candies_of(20, 4, 1) == 7
for c in range(1, 300):
    for n in [1, 2, 3, 5, 8]:
        sim = distribute_simulation(c, n)
        assert all(answer_candies_of(c, n, i) == sim[i] for i in range(n))
        # người nhận cuối = người có lần trao cuối trong mô phỏng
        t, left = 0, c
        while left > 0:
            last = t % n; left -= min(t + 1, left); t += 1
        assert answer_last_receiver(c, n) == last
        for d in [1, 2, 3]:
            assert answer_distribute_step_d_math(c, n, d) == distribute_step_d(c, n, d), (c, n, d)
print("Đáp án tham khảo: PASS toàn bộ")

Đáp án tham khảo: PASS toàn bộ


## 18. Template nhớ nhanh + checklist submit

```python
# Mô phỏng (đủ nhanh vì chỉ ~√(2c) lần trao)
ans = [0] * n
t = 0
while candies > 0:
    give = min(t + 1, candies)
    ans[t % n] += give
    candies -= give
    t += 1

# Công thức
p = (math.isqrt(8 * candies + 1) - 1) // 2     # số lần trao đầy đủ
rows, cols = divmod(p, n)
ans[i] = rows*(i+1) + n*rows*(rows-1)//2 + (rows*n + i + 1 if i < cols else 0)
ans[p % n] += candies - p*(p+1)//2
```

**Checklist:**
- [ ] Lượng trao **tăng liên tục** qua các vòng (không reset về 1)?
- [ ] Người cuối chỉ nhận phần còn lại (`min`)?
- [ ] Chỉ số quay vòng `t % n`?
- [ ] Dùng `//` và `math.isqrt`, không dùng `/`, `math.sqrt`?
- [ ] Tổng kết quả bằng `candies`? Độ dài bằng `num_people`?
- [ ] Trường hợp `candies < num_people` (nhiều người nhận 0)?